In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

In [2]:
movies_df = pd.read_csv('../WBSflix_dataset/ml-latest-small/movies.csv')
ratings_df = pd.read_csv('../WBSflix_dataset/ml-latest-small/ratings.csv')

# Exploration of DFs

## movies

In [3]:
movies_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  9742 non-null   int64 
 1   title    9742 non-null   object
 2   genres   9742 non-null   object
dtypes: int64(1), object(2)
memory usage: 228.5+ KB


In [4]:
movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [5]:
movies_df.isna().any()

movieId    False
title      False
genres     False
dtype: bool

## ratings

In [6]:
ratings_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


In [7]:
ratings_df.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [8]:
ratings_df.isna().any()

userId       False
movieId      False
rating       False
timestamp    False
dtype: bool

# 1. Recommend top rated movies

Create a function that takes as an input the dataframes “ratings” and “movies” and outputs the top n movies so that you can populate the first row of the site, using your hybrid system.

In [9]:
def popular_movies():
    # create dataframe with movieId and average rating
    ratings_pure_df = pd.DataFrame(ratings_df.drop(['userId', 'timestamp'], axis=1).groupby('movieId')['rating'].mean())

    # create dataframe with movieId and number of people who have reviewed that movie
    no_of_ratings_df = pd.DataFrame(ratings_df.drop(['rating', 'timestamp'], axis=1).groupby('movieId')['userId'].count())
    # change column name to be more descriptive
    no_of_ratings_df.rename(columns={'userId': 'no_of_reviews'}, inplace=True)

    # merge all dataframes together
    movie_rating = ratings_pure_df.merge(movies_df, left_on='movieId', right_on='movieId').merge(no_of_ratings_df, left_on='movieId', right_on='movieId')

    # movies with more than 50 reviews in order of rating
    over_50 = movie_rating[movie_rating['no_of_reviews'] >= 50].sort_values('rating', ascending=False)
    
    # return columns in easier to read order
    return over_50[['title', 'rating', 'no_of_reviews']]

In [10]:
popular_movies()

,title,rating,no_of_reviews
277,"Shawshank Redemption, The (1994)",4.429022,317
659,"Godfather, The (1972)",4.289062,192
2224,Fight Club (1999),4.272936,218
974,Cool Hand Luke (1967),4.271930,57
602,Dr. Strangelove or: How I Learned to Stop Worr...,4.268041,97
...,...,...,...
144,Johnny Mnemonic (1995),2.679245,53
145,Judge Dredd (1995),2.669355,62
376,City Slickers II: The Legend of Curly's Gold (...,2.645455,55
379,Coneheads (1993),2.420635,63


# 2. Recommend based on another movie watched by the user

Create a function that takes as an input a movie that a user has liked or watched, and outputs the top n most similar movies so that you can populate the second row of the site

## 2a. movies that people who also watched the selected movie watched - ranked by amount of ratings

In [11]:
def what_we_also_watched(movie_title, n):
    
    # merge ratings & movies DFs
    rating_movies = ratings_df.merge(movies_df, left_on='movieId', right_on='movieId')

    # select only rows containing inputed movie
    single_movie = rating_movies[rating_movies['title'] == movie_title]

    # left join movies list based on userids of inputed movie
    # this shows us what the users who watched our inputed movie have also watched
    user_results = pd.merge(single_movie, rating_movies, on='userId', how='left')

    # count the number of times each movie occurs in list of also watched movies
    user_results_counted_df = pd.DataFrame(user_results.groupby('title_y')['userId'].count())
    # change column name to be more descriptive
    user_results_counted_df.rename(columns={'userId': 'no_of_times_also_watched'}, inplace=True)

    # sort dataframe by number of times the movie appears, plus drop inputed movie from the list
    also_watched_df = user_results_counted_df.sort_values('no_of_times_also_watched', ascending=False).drop(movie_title)

    # return n number of movies
    return also_watched_df.head(n)

In [12]:
what_we_also_watched('Pocahontas (1995)', 6)

,no_of_times_also_watched
title_y,
"Lion King, The (1994)",54
Aladdin (1992),52
Beauty and the Beast (1991),50
Forrest Gump (1994),50
Jurassic Park (1993),46
Toy Story (1995),42


## 2b. using Pearsons correlation to find similar movies

In [31]:
def pearson_recommends(movie_name_, n):
    
    # i. find movie_id, name's can be confusing
    find_id_df = movies_df[movies_df.title.str.contains(movie_name_)].reset_index()
    movie_id_ = find_id_df.movieId[0]
    
    # ii. pivot table
    places_crosstab = pd.pivot_table(data=ratings_df, values='rating', index='userId', columns='movieId')

    # iii. select column for chosen movie
    top_ratings = places_crosstab[movie_id_]
    # select only rows with a rating (no NaNs)
    top_ratings.dropna(inplace=True)

    # iv. compare selected movie column with whole table to get correlations
    corr_top = pd.DataFrame(places_crosstab.corrwith(top_ratings), columns=['PearsonR'])
    # exlude NaNs
    corr_top.dropna(inplace=True)

    # v. count the number of times each movie occurs in list of also watched movies
    user_reviews_counted_df = pd.DataFrame(ratings_df.groupby('movieId')['userId'].count())
    # change column name to be more descriptive
    user_reviews_counted_df.rename(columns={'userId': 'no_of_times_reviewed'}, inplace=True)
    
    # vi. merge iv. & v. (add review count to correlations)
    top_corr_summary = corr_top.merge(user_reviews_counted_df, left_on='movieId', right_on='movieId')
    
    # vii. filter out movies with less than 10 reviews
    top_n = top_corr_summary[top_corr_summary['no_of_times_reviewed']>=10].sort_values('PearsonR', ascending=False)

    # viii. left merge movies_df to get movie titles
    top_n = pd.merge(top_n, movies_df, on='movieId', how='left').drop('genres', axis=1)
    
    # ix. return top n number of movies
    return top_n.head(n)

In [32]:
pearson_recommends('Last Man', 6)

/opt/anaconda3/lib/python3.8/site-packages/numpy/lib/function_base.py:2634: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/opt/anaconda3/lib/python3.8/site-packages/numpy/lib/function_base.py:2493: RuntimeWarning: divide by zero encountered in true_divide
  c *= np.true_divide(1, fact)


,movieId,PearsonR,no_of_times_reviewed,title
0,2642,1.0,22,Superman III (1983)
1,1952,1.0,22,Midnight Cowboy (1969)
2,62374,1.0,11,Body of Lies (2008)
3,5481,1.0,65,Austin Powers in Goldmember (2002)
4,5505,1.0,12,"Good Girl, The (2002)"
5,5528,1.0,19,One Hour Photo (2002)


## 2c. Using cosine similarity (of ratings in database) to predict similar movies

In [37]:
def cos_recommender(movie_name_, n):
    
    # find movie_id, name's can be confusing
    find_id_df = movies_df[movies_df.title.str.contains(movie_name_)].reset_index()
    movie_id_ = find_id_df.movieId[0]
    
    # create df containg only movieId and average rating
    pure_ratings_df = pd.DataFrame(ratings_df.drop(['userId', 'timestamp'], axis=1).groupby('movieId')['rating'].mean().reset_index().copy())
    
    # create cosine similarity table
    sample_similarity = pd.DataFrame(cosine_similarity(pure_ratings_df), columns=pure_ratings_df.movieId, index=pure_ratings_df.movieId)
    
    # create list of cos similarities to inputed movie. Remove inputed movie from list
    movie_recommendations = pd.DataFrame(sample_similarity[movie_id_].drop(movie_id_).reset_index().copy())
    
    # rename columns for ease of understanding
    movie_recommendations.rename(columns={movie_id_: 'cos_similarity'}, inplace=True)
    
    # merge with movies_df to get titles of movies
    movie_recommendations_with_names = movie_recommendations.merge(movies_df, left_on='movieId', right_on='movieId')
    
    # rank list by cos similarity
    recommendations_ordered = movie_recommendations_with_names.sort_values(by=['cos_similarity'], ascending=False)
    
    # return top n entries
    return recommendations_ordered.head(n)

In [38]:
cos_recommender('Jumanji', 6)

,movieId,cos_similarity,title,genres
1,3,0.976702,Grumpier Old Men (1995),Comedy|Romance
0,1,0.961621,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
4,6,0.895440,Heat (1995),Action|Crime|Thriller
3,5,0.881259,Father of the Bride Part II (1995),Comedy
2,4,0.872437,Waiting to Exhale (1995),Comedy|Drama|Romance
5,7,0.816134,Sabrina (1995),Comedy|Romance


# 3. Recommend by predicting users taste, based on weighted movie ratings (ratings weighted by cosine similarit of users)

## 3a. Movies from most similar user (1st iteration)

In [41]:
def recommendations_based_on_most_similar_user(user_id_, n):
    
    # create pivot table
    movie_ratings_crosstab = pd.pivot_table(data=ratings_df, values='rating', index='userId', columns='movieId')
    
    # all NaN to 0
    mrct_no_no_no = movie_ratings_crosstab.fillna(0)
    
    # compare cos similarity of users
    ratings_distances = pd.DataFrame(cosine_similarity(mrct_no_no_no), columns=mrct_no_no_no.index, index=mrct_no_no_no.index)
    
    # create list of cos similarities to inputed user. Remove inputed user from list
    similar_users = pd.DataFrame(ratings_distances[user_id_].drop(user_id_).reset_index())
    
    # rename columns for ease of understanding
    similar_users.rename(columns={user_id_: 'cos_similarity'}, inplace=True)
    
    # rank list by cos similarity
    users_ordered = similar_users.sort_values(by=['cos_similarity'], ascending=False)
    
    # get user id of most similar user
    top_ = users_ordered.iloc[:1,:].reset_index()
    most_similar_user = top_.userId[0]
    
    # get movies reviewed by most similar user, order by rating
    most_similar_reviews = ratings_df[ratings_df['userId'] == most_similar_user].sort_values(by=['rating'], ascending=False)
    
    # merge with movies_df to get titles of movies
    most_similar_w_titles = pd.merge(most_similar_reviews, movies_df, on='movieId', how='left')
    
    # return with n number of results
    return most_similar_w_titles.head(n)

In [42]:
recommendations_based_on_most_similar_user(3, 6)

,userId,movieId,rating,timestamp,title,genres
0,313,1274,5.0,1030474742,Akira (1988),Action|Adventure|Animation|Sci-Fi
1,313,3740,5.0,1030556316,Big Trouble in Little China (1986),Action|Adventure|Comedy|Fantasy
2,313,2117,5.0,1030474778,1984 (Nineteen Eighty-Four) (1984),Drama|Sci-Fi
3,313,3550,5.0,1030475654,The Hunger (1983),Horror
4,313,3578,5.0,1030556236,Gladiator (2000),Action|Adventure|Drama
5,313,968,5.0,1030474778,Night of the Living Dead (1968),Horror|Sci-Fi|Thriller


## 3b. Movies by weight - each users cos similarity creates a weight, and then multiply their rating for a movie by this weight (higher ratings by similar users have more influence) (1st iteration)

In [53]:
def recommendations_based_on_weight(user_id_, n):
    
    # create pivot table
    movie_ratings_crosstab = pd.pivot_table(data=ratings_df, values='rating', index='userId', columns='movieId')
    
    # all NaN to 0
    mrct_no_no_no = movie_ratings_crosstab.fillna(0)
    
    # compare cos similarity of users
    ratings_distances = pd.DataFrame(cosine_similarity(mrct_no_no_no), columns=mrct_no_no_no.index, index=mrct_no_no_no.index)
    
    # create list of cos similarities to inputed user. Remove inputed user from list
    similar_users = pd.DataFrame(ratings_distances[user_id_].drop(user_id_).reset_index().copy())
    
    # rename columns for ease of understanding
    similar_users.rename(columns={user_id_: 'cos_similarity'}, inplace=True)
    
    # make userId index, so there's no confusion when finding weights
    similar_users.set_index('userId', inplace = True)
    
    # find weight of each user
    weights_df = pd.DataFrame(similar_users.cos_similarity / sum(similar_users.cos_similarity))
    # rename column to better understand
    weights_df.rename(columns={'cos_similarity':'weight'}, inplace=True)
    
    # Now time to find movies not seen by user
    seen_movies = ratings_df[ratings_df.userId == user_id_].copy()
    
    # movieId as index, to make using .drop easier
    seen_movies.set_index('movieId', inplace = True)
    
    # set movieId as index in list of complete movies
    new_index_movies = movies_df.set_index('movieId').copy()
    
    # drop seen movies from movie list
    not_seen_movies = new_index_movies.drop(seen_movies.index, axis=0).copy()
    
    # left merge ratings on to not_seen_movies
    not_seen_ratings = pd.merge(not_seen_movies, ratings_df, on='movieId', how='left')
    
    # left merge weights onto df created in last step
    not_seen_weights = pd.merge(not_seen_ratings, weights_df, on='userId', how='left')
    
    # create column for rating multiplied by weight
    not_seen_weights['rating_x_weight'] = not_seen_weights.rating * not_seen_weights.weight
    
    # group by movie, and sum rating_x_weight
    estimated_rating = not_seen_weights.groupby('movieId')['rating_x_weight'].sum().copy()
    estimated_rating_df = pd.DataFrame(estimated_rating.reset_index())
    
    # join movie names
    estimated_rating_df = estimated_rating_df.merge(movies_df, on='movieId', how='left')
    
    # order by rating_x_weight
    estimated_rating_df.sort_values(by=['rating_x_weight'], ascending=False, inplace=True)
    
    # return n number of movies
    return estimated_rating_df.head(n)

In [54]:
recommendations_based_on_weight(80, 5)

,movieId,rating_x_weight,title,genres
310,356,2.795524,Forrest Gump (1994),Comedy|Drama|Romance|War
254,296,2.688944,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller
221,260,2.355542,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi
3595,4993,2.274031,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy
4735,7153,2.223860,"Lord of the Rings: The Return of the King, The...",Action|Adventure|Drama|Fantasy


## 3c. Testing the accuracy of the predictions (3rd iteration)

In [63]:
# create pivot table
movie_ratings_crosstab = pd.pivot_table(data=ratings_df, values='rating', index='userId', columns='movieId')

# all NaN to 0
mrct_no_no_no = movie_ratings_crosstab.fillna(0)

# compare cos similarity of users
ratings_distances = pd.DataFrame(cosine_similarity(mrct_no_no_no), columns=mrct_no_no_no.index, index=mrct_no_no_no.index)

In [56]:
# create a cop to train the predictions on
train = mrct_no_no_no.copy()

In [64]:
# remove 2 known predicitons to test the accurac of the predictions against
train.iloc[0,0] = 0 # user 1, movie1 (was 4.0)
train.iloc[5,2] = 0 # user 6, movie 3 (was 5.0)

In [65]:
import numpy as np
test = (
    pd.DataFrame(
        np.zeros(mrct_no_no_no.shape), # create a matrix full of 0 of the same shape than the sample
        columns=mrct_no_no_no.columns, index=mrct_no_no_no.index
    )
    .apply(pd.to_numeric, downcast='integer')
)
test.iloc[0,0] = mrct_no_no_no.iloc[0,0]
test.iloc[5,2] = mrct_no_no_no.iloc[5,2]

In [59]:
train_similarity = pd.DataFrame(cosine_similarity(train), columns=mrct_no_no_no.index, index=mrct_no_no_no.index)

user_1_pred = sum(train[1] * train_similarity[1]) / (sum(train_similarity[1]) - 1)
user_6_pred = sum(train[3] * train_similarity[6]) / (sum(train_similarity[6]) - 1)
print(
    f"""
    User 1 rating for movie 1 was {mrct_no_no_no.iloc[0,0]} and their prediction is {user_1_pred}, 
    User 6 rating for movie 3 was {mrct_no_no_no.iloc[5,2]} and their prediction is {user_6_pred} 
    """
)


    User 1 rating for movie 1 was 4.0 and their prediction is 1.7453662937564258, 
    User 6 rating for movie 3 was 5.0 and their prediction is 0.5072958929559568 
    


## 3d. predicting using closest neighbours (4th iteration)

In [61]:
train = mrct_no_no_no.copy()
train.iloc[0,0] = 0 # user 1, movie1 (was 4.0)
train.iloc[5,2] = 0 # user 6, movie 3 (was 5.0)
test = (
    pd.DataFrame(
        np.zeros(mrct_no_no_no.shape), # create a matrix full of 0 of the same shape than the sample
        columns=mrct_no_no_no.columns, index=mrct_no_no_no.index
    )
    .apply(pd.to_numeric, downcast='integer')
)
test.iloc[0,0] = mrct_no_no_no.iloc[0,0]
test.iloc[5,2] = mrct_no_no_no.iloc[5,2]

# train the model
train_similarity = pd.DataFrame(cosine_similarity(train), columns=mrct_no_no_no.index, index=mrct_no_no_no.index)

# select only the closer neigbours 
weighted_ratings = (
pd.DataFrame({'ratings': train[3],
              'similarities': train_similarity[6]})
    .sort_values('similarities', ascending=False) # order values with higher similarities
    .head(5) # select user 6 + the 4 closer neigbours
    .query('ratings != 0') # filter the similarity 1, which is the own student
    .assign(weighted_ratings = lambda x: x.ratings * x.similarities) # weight the food and similarities
)
print(weighted_ratings)
# calculate the ponderated weight
user_6_pred = sum(weighted_ratings.weighted_ratings) / sum(weighted_ratings.similarities)
print(
    f"""
    User 6 rating was {mrct_no_no_no.iloc[5,2]} and their predictions is {user_6_pred}, 
    """
)

        ratings  similarities  weighted_ratings
userId                                         
117         3.0      0.566647          1.699940
58          3.0      0.521872          1.565616

    User 6 rating was 5.0 and their predictions is 3.0, 
    


## 3e. creating a test/train split with sklearn (5th iteration)

In [78]:
# 1 Merge ratings & movies
data = ratings_df.merge(movies_df, on="movieId", how="left")

# 2 Create the big users-movies table
movie_user = data.pivot_table(index='userId',columns='title',values='rating')
movie_user.fillna(0, inplace=True)

1. find all ratings that aren't 0

In [79]:
ratings_pos = pd.DataFrame(
    np.nonzero(np.array(movie_user)) # find out all the positions different than 0
).T

# first column represents the index position, the second = column position
ratings_pos.head(20)

,0,1
0,0,48
1,0,66
2,0,202
3,0,245
4,0,325
5,0,327
6,0,346
7,0,405
8,0,420
9,0,440


2. create a test/train

In [80]:
# split with train and test
train_pos, test_pos = train_test_split(ratings_pos, random_state=42, test_size=.2)

In [81]:
# create an empty dataframe full of 0, with the same shape as the food data
train = np.zeros(movie_user.shape)
# fill the set with the ratings based on the train positions
for pos in train_pos.values: 
    index = pos[0]
    col = pos[1]
    train[index, col] = movie_user.iloc[index, col]
train = pd.DataFrame(train, columns=movie_user.columns, index=movie_user.index).apply(pd.to_numeric, downcast='integer')

In [82]:
# now it is time for the test set. We will follow the same process
test = np.zeros(movie_user.shape)
for pos in test_pos.values: 
    index = pos[0]
    col = pos[1]
    test[index, col] = movie_user.iloc[index, col]
test = pd.DataFrame(test, columns=movie_user.columns, index=movie_user.index).apply(pd.to_numeric, downcast='integer')

3. train model to predict all ratings

In [85]:
# train the model
train_similarity = pd.DataFrame(cosine_similarity(train), columns=movie_user.index, index=movie_user.index)

In [86]:
def recommender(index_name, column_name, sim_df, data): 
    results = (
    pd.DataFrame({
        'ratings': data.loc[:,column_name], 
        'similarities' : sim_df.loc[index_name,:].tolist()
    })
        .assign(weighted_ratings = lambda x: x.ratings * x.similarities)
        .query('ratings != 0')
        .agg({
            'weighted_ratings':'sum', 
            'similarities':'sum'
        })
    )
    if results[1] == 0:
        return 9001
    pred_rating = results[0] / results[1]
    return pred_rating

In [77]:
# ***** This section produces no results and runs for ages due to a double_scaler
# ***** double_scaler = dividing by 0
# ***** this needs to be resolved to continue

# import numpy as np

# np.seterr(all='raise')

# recommendations = pd.DataFrame(np.zeros(movie_user.shape), columns=movie_user.columns, index=movie_user.index)

# for col in train.columns: 
#     for index in train.index:
#         try:
#             recommendations.loc[index, col] = round(recommender(index, col, train_similarity, train),1)
#         except FloatingPointError:
#             print(index, col)
# recommendations

KeyboardInterrupt: 